In [9]:
import torch
import sys
import os
PROJECT_ROOT = r"C:\\Dev\\Synthetic-Data-Generation-For-Cardiovascular-Disease-Risk-Prediction\\"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from torchinfo import summary
from UpsampleAndConv1D import Generator, Critic

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
generator = Generator()
generator.to(device)
with torch.autograd.no_grad():
    dummy_input = torch.randn(1,100, device=device)
    cond = torch.tensor([[0]],dtype=torch.long, device=device)
    output = generator(dummy_input,cond)

cuda


In [11]:
print(os.getcwd())
metrics = torch.load("models/UpsampleAndCNN_CWGAN/Model_1_GP_10.0_DTW_1.0/Model.pth", weights_only=False)

c:\Dev\Synthetic-Data-Generation-For-Cardiovascular-Disease-Risk-Prediction\Final_models\CWGAN


In [ ]:
critic = Critic()
critic.load_state_dict(metrics['critic_state_dict'])
critic.eval()
critic.to(device)

c:\Dev\Synthetic-Data-Generation-For-Cardiovascular-Disease-Risk-Prediction\.venv\Lib\site-packages\torch\nn\modules\conv.py:366: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1032.)
  return F.conv1d(


Critic(
  (cond_block): Sequential(
    (0): Embedding(4, 32)
    (1): Linear(in_features=32, out_features=32, bias=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (block1): CriticConvBlock(
    (conv1): Conv1d(3, 32, kernel_size=(16,), stride=(1,), padding=same)
    (conv2): Conv1d(32, 64, kernel_size=(16,), stride=(1,), padding=same)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (act): LeakyReLU(negative_slope=0.3, inplace=True)
  )
  (block2): CriticConvBlock(
    (conv1): Conv1d(64, 128, kernel_size=(16,), stride=(1,), padding=same)
    (conv2): Conv1d(128, 256, kernel_size=(16,), stride=(1,), padding=same)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (act): LeakyReLU(negative_slope=0.3, inplace=True)
  )
  (film1): FiLM1D(
    (net): Sequential(
      (0): Linear(in_features=32, out_features=64, bias=True)
      (1): LeakyReLU(negative_slope=0.2, inplace=True)
      (2): Lin

In [19]:
summary(critic, input_data=(output,cond))

Layer (type:depth-idx)                   Output Shape              Param #
Critic                                   [1, 1]                    --
├─Sequential: 1-1                        [1, 32]                   --
│    └─Embedding: 2-1                    [1, 32]                   128
│    └─Linear: 2-2                       [1, 32]                   1,056
│    └─LeakyReLU: 2-3                    [1, 32]                   --
├─CriticConvBlock: 1-2                   [1, 64, 320]              --
│    └─Conv1d: 2-4                       [1, 32, 640]              1,568
│    └─LeakyReLU: 2-5                    [1, 32, 640]              --
│    └─Conv1d: 2-6                       [1, 64, 640]              32,832
│    └─LeakyReLU: 2-7                    [1, 64, 640]              --
│    └─MaxPool1d: 2-8                    [1, 64, 320]              --
├─FiLM1D: 1-3                            [1, 64, 320]              --
│    └─Sequential: 2-9                   [1, 128]                  --
│   